## Conform model

In [8]:
num_list = []

with open('policy_document.txt', 'r') as fh:
    for line in fh:
        num_list.append(int(line))

ValueError: invalid literal for int() with base 10: 'policy_documents = [\n'

In [ ]:
import json
import os
import time
from typing import List, Dict, Any
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
from transformers import pipeline

#======
# Pipeline & wrappeur
#======

def extract_code_cells_from_notebook(input_data):
    """
    Args:
        input_data (str): Either a path to a .ipynb file OR a raw code string.

    Returns:
        list: A list of code strings.
    """
    # 1. Check if the input is a path to a notebook file
    if isinstance(input_data, str) and input_data.endswith('.ipynb') and os.path.exists(input_data):
        try:
            with open(input_data, 'r', encoding='utf-8') as f:
                notebook = json.load(f)
            
            code_blocks = []
            for cell in notebook.get('cells', []):
                if cell.get('cell_type') == 'code':
                    source = cell.get('source', [])
                    # Notebook source can be a list of lines or a single string
                    code = ''.join(source) if isinstance(source, list) else source
                    if code.strip():  # Only add non-empty cells
                        code_blocks.append(code)
            return code_blocks
            
        except (json.JSONDecodeError, KeyError):
            # If it's a .ipynb but malformed, treat it as a raw string or raise error
            raise ValueError(f"File at {input_data} is not a valid Jupyter Notebook.")
    
    # 2. If it's not a notebook path, treat the entire string as a single code chunk
    else:
        # Wrap in a list to keep the return type consistent
        return [input_data] if input_data.strip() else []


def load_and_chunk_documents():
    """
    Load sample notebooks example classfication and chunk them for better retrieval.
    
    """
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    policy_documents = [
        {
            "desc": "Classic ML fit (should detect Model Training)",
        "code": """ 
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)"""
        },
        {
             "desc": "Data loading (should detect Data Loading)",
        "code": """
import pandas as pd
df = pd.read_csv('data.csv')
"""
        },
        {
             "desc": "Model evaluation (should detect Model Evaluation)",
        "code": """
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred))
"""
        },
        {
            "desc": "Deep learning training (should detect Model Training, Deep Learning)",
        "code": """
import tensorflow as tf
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(10))
model.compile(optimizer='adam', loss='mse')
model.fit(X, y, epochs=5)
"""
        },
        {
            "desc": "Empty code (should return nothing or fallback)",
        "code": ""
        },
        {
            "desc": "Non-ML code (should not hallucinate ML steps)",
        "code": """
print("Hello, world!")
a = 5 + 3
"""
        },
        {
            "desc": "Multiple ML steps in one cell (should detect all present steps)",
        "code": """
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
X_train, X_test, y_train, y_test = train_test_split(X, y)
model = RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
"""
        },
        {
            "desc": "Unusual ML library (should fallback or try to detect)",
        "code": """
import xgboost as xgb
model = xgb.XGBClassifier()
model.fit(X_train, y_train)
"""
        },
        {
            "desc": "Malformed code (should not crash, should fallback)",
        "code": """
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)
"""
        },
        {
            "desc": "Comments only (should fallback)",
        "code": """
# This cell only contains comments
# No executable code here"""
        },
        {
            "desc": "Irrelevant imports (should not hallucinate ML steps)",
        "code": """
import os
import sys
print(os.getcwd())
"""
        },
        {
            "desc": "Ambiguous variable names (should not guess)",
        "code": """
foo = bar(X, y)
baz = foo.predict(X)
"""

        },
        {
            "desc": "Only function definitions (should fallback)",
        "code": """
def my_function(x):
    return x * 2
"""

        },
        {
            "desc": "ML and non-ML mixed (should only detect ML parts)",
        "code": """
import pandas as pd
df = pd.read_csv('data.csv')
print("Loaded data")
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(df['X'], df['y'])
print("Done")
"""

        }
    ]
    
    print(f"Loaded notebook code examples classification successful")
    
    # Configure text splitter 
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", " ", ""]
    )
    
    # Chunk all documents
    all_chunks = []
    for doc in policy_documents:
        chunks = text_splitter.split_text(doc["code"])
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "desc": doc["desc"],
                "code": chunk,
            })
    
    return all_chunks


def setup_vector_database(chunks):
    """
    Set up ChromaDB vector database and store document chunks.
    
    """
    import chromadb
    """ 
    
    try:
        client.delete_collection(name="notebook_examples_policy")
    except:
        pass
    """
    # Initialize ChromaDB client
    client = chromadb.Client()
    
    # Create collection or get it
    
    collection = client.get_or_create_collection(name="notebook_examples_policy")#  collection name with underscores and no spaces
    
    
    # Prepare data for storage
    ids = [f"chunk_{i}" for i in range(len(chunks))]
    codes = [chunk["code"] for chunk in chunks]
    descriptions = [{"desc": chunk["desc"]} for chunk in chunks]
    
    # Adding documents to collection & embedding
    collection.add(
        ids=ids,
        documents=codes,
        metadatas=descriptions
    )
        
    print("Vector Database Created")
    return collection

def process_user_query(query: str):
    """
    Process user query and convert to embedding for vector search.
    
    """
    from sentence_transformers import SentenceTransformer
    
    # Loading embedding model 
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Preprocess query
    cleaned_query = str(query).lower().strip()
    
    # Convert query to embedding
    query_embedding = model.encode(cleaned_query)

    return query_embedding


def search_vector_database(collection, query_embedding, top_k: int = 3):
    """
    Search vector database for relevant document chunks.
    """
    
    # Performing vector search
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )
    
    # Process results
    search_results = []
    for i in range(len(results['ids'][0])):
        doc_id = results['ids'][0][i]
        distance = results['distances'][0][i]
        document = results['documents'][0][i]
        metadata = results['metadatas'][0][i]
        
        similarity = 1 - distance
        search_results.append({
            'id': doc_id,
            'content': document,
            'metadata': metadata,
            'similarity': similarity
        })
    
    return search_results


def augment_prompt_with_context(query: str, search_results: List[Dict]) -> str:
    """
    Build augmented prompt with retrieved context for LLM.

    """
    # Assemble context from search results
    context_parts = []
    for i, result in enumerate(search_results, 1):
        context_parts.append(f"Source {i}: {result['metadata']}\n{result['content']}")
    
    context = "\n\n".join(context_parts[:5]) #max 5 chunks
    
    
    # Build augmented prompt
    augmented_prompt = f"""
You are an assistant. Answer using only the context, and based on the following notebook chunk code classification examples, classify all the code chunks on this notebook.

Context:
{context}

Given notebook:
{query}

Answer:
"""
    
    return augmented_prompt

from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="bigcode/starcoder2-3b",
    device_map="auto",
    load_in_4bit=True,
    trust_remote_code=True
)

def generate_response(augmented_prompt):
    
    """Generating an answer for the classification task by using an LLM

    Returns:
        _type_: asnwer from the LLM (string)
    """
    from transformers import pipeline
    """ 
    generator = pipeline(
        "text-generation",
        model='mistralai/Mistral-7B-Instruct-v0.2'  # or 'google/flan-t5-large'
    )
    """
    
    response = generator(
        augmented_prompt,
        max_new_tokens=4000,
        temperature=0.0,
        do_sample=False
    )
    
    return response[0]["generated_text"]

def rag_pipeline(query01):
    """
    Running the complete RAG pipeline from start to finish.
    Here the query parameter is the notebook we put in entry, for it to be classified by a RAG pepiline using starcoder2-3b.
    The query01 input is a raw string to where the notebook in json is located, for it to be preprocessed and input into the RAG pepeline
    """
    #step0

    query = extract_code_cells_from_notebook(query01)

    # Step 1: Load and chunk documents
    chunks = load_and_chunk_documents()
    
    # Step 2: Setup vector database
    collection = setup_vector_database(chunks)
    print(f"DEBUG - collection = {collection}")  
    print(f"DEBUG - type = {type(collection)}")
    # Step 3: Process user query
    query_embedding = process_user_query(query)
    print(f"DEBUG - embedding shape: {query_embedding.shape}")
    # Step 4: Search vector database
    search_results = search_vector_database(collection, query_embedding)
    
    # Step 5: Augment prompt with context
    augmented_prompt = augment_prompt_with_context(query, search_results)
    
    # Step 6: Generate response
    response = generate_response(augmented_prompt)
    
    # Display final result
    print("FINAL RESULT\n")
    
    
    return response



c:\Users\arion\rag-classification\standard_rag\RAG_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\arion\rag-classification\standard_rag\RAG_env\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [1]:
# END

!pip freeze > requirements.txt